In [1]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
df = pd.read_parquet("../data/processed/tweets_ai_clean.parquet")

df.head(10)

,id,conversation_id,created_at,date,time,timezone,tweet,language,urls,photos,...,cashtags,retweet,quote_url,video,thumbnail,dominios_url,tipo_fuente_url,tweet_clean,tweet_clean_filt,num_palabras_filt
0,826580756675694720,826580756675694720,2017-01-31 18:59:44,2017-01-31,18:59:44,-400,What topic could spark human dialogue on AI? Artificial Intelligence Glossary for Marketers https://t.co/SPyvplngtg @hubspot @Megan_Conley_,en,[https://blog.hubspot.com/marketing/artificial-intelligence-glossary-marketers],[],...,[],False,None,0,None,[blog.hubspot.com],[news],what topic could spark human dialogue on ai artificial intelligence glossary for marketers,what topic could spark human dialogue on ai artificial intelligence glossary for marketers,13
1,826580719921152000,826580719921152000,2017-01-31 18:59:35,2017-01-31,18:59:35,-400,"I think #ArtificialIntelligence will augment humanity, way before it displaces it. #AI #ChasmWaxing https://t.co/SAsmyqEe2b",en,[http://bit.ly/2bWb1ly],[],...,[],False,None,0,None,[bit.ly],[other],i think artificialintelligence will augment humanity way before it displaces it ai chasmwaxing,i think will augment humanity way before it displaces it,10
2,826580631366700928,826580631366700928,2017-01-31 18:59:14,2017-01-31,18:59:14,-400,What Managers Need to Know About Artificial Intelligence https://t.co/t6VjDcGONz via @mitsmr,en,[http://mitsmr.com/2klQ17D],[],...,[],False,None,0,None,[mitsmr.com],[news],what managers need to know about artificial intelligence via,what managers need to know about artificial intelligence via,9
3,826580578954780672,826580578954780672,2017-01-31 18:59:01,2017-01-31,18:59:01,-400,4 ways Artificial Intelligence changes the game for SEO https://t.co/uJnUtUSa2E #thenextweb,en,[http://bit.ly/2koUCIB],[],...,[],False,None,0,None,[bit.ly],[other],4 ways artificial intelligence changes the game for seo thenextweb,4 ways artificial intelligence changes the game for seo,9
4,826580577457287040,826580577457287040,2017-01-31 18:59:01,2017-01-31,18:59:01,-400,Artificial Intelligence can Shape the Future of Society https://t.co/LOPLuWPt8W | follow @whyfuture_AI,en,[https://www.youtube.com/watch?v=NQefTsrSjJY],[],...,[],False,None,0,None,[youtube.com],[social],artificial intelligence can shape the future of society follow,artificial intelligence can shape the future of society follow,9
5,826580175248699520,826580175248699520,2017-01-31 18:57:25,2017-01-31,18:57:25,-400,IBM #Watson uses artificial intelligence to create rum that tastes like a holiday https://t.co/XtxV3aYADa https://t.co/Lg8Lc0KOGX,en,[http://bit.ly/2kO88U7],[https://pbs.twimg.com/media/C3iaSR9UEAAEDW5.jpg],...,[],False,None,1,https://pbs.twimg.com/media/C3iaSR9UEAAEDW5.jpg,[bit.ly],[other],ibm watson uses artificial intelligence to create rum that tastes like a holiday,ibm uses artificial intelligence to create rum that tastes like a holiday,12
6,826580141543415808,826580141543415808,2017-01-31 18:57:17,2017-01-31,18:57:17,-400,How Artificial Intelligence Will Revolutionize eDiscovery https://t.co/elT5C0xm0N https://t.co/OvoqxjkvgK,en,[http://upflow.co/l/45Ug],[https://pbs.twimg.com/media/C3iaQfkWQAAqKDv.jpg],...,[],False,None,1,https://pbs.twimg.com/media/C3iaQfkWQAAqKDv.jpg,[upflow.co],[other],how artificial intelligence will revolutionize ediscovery,how artificial intelligence will revolutionize ediscovery,6
7,826580101148135424,826580101148135424,2017-01-31 18:57:07,2017-01-31,18:57:07,-400,How Artificial Intelligence Will Kickstart the Internet of Things #Nitaipartners #internetofthings #bigdataanalytic… https://t.co/Hcp6VPQF4C,en,[http://buff.ly/2kJMRht],[],...,[],False,None,0,None,[buff.ly],[other],how artificial intelligence will kickstart the internet of things nitaipartners internetofthings bigdataanalytic,how artificial intelligence will kickstart the internet of things,9
8,826580083519397760,826580083519397760,2017-01-31 18:57:03,2017-01-31,18:57:03,-400,This man is not mentally all there. That or he's actually artificia

### Preprocesado para huggingface
Es un preprocesado minimo sustituyendo los usuarios por @user y las URLs por http. 

In [2]:
def preprocess_for_hf(text):
    if not isinstance(text, str):
        return ""

    tokens = []
    for tok in text.split():
        if tok.startswith("@") and len(tok) > 1:
            tok = "@user"
        elif tok.startswith("http"):
            tok = "http"
        tokens.append(tok)

    return " ".join(tokens)

df["tweet_hf"] = df["tweet"].apply(preprocess_for_hf)

In [3]:
df[["tweet", "tweet_hf"]].head(3)

,tweet,tweet_hf
0,What topic could spark human dialogue on AI? Artificial Intelligence Glossary for Marketers https://t.co/SPyvplngtg @hubspot @Megan_Conley_,What topic could spark human dialogue on AI? Artificial Intelligence Glossary for Marketers http @user @user
1,"I think #ArtificialIntelligence will augment humanity, way before it displaces it. #AI #ChasmWaxing https://t.co/SAsmyqEe2b","I think #ArtificialIntelligence will augment humanity, way before it displaces it. #AI #ChasmWaxing http"
2,What Managers Need to Know About Artificial Intelligence https://t.co/t6VjDcGONz via @mitsmr,What Managers Need to Know About Artificial Intelligence http via @user


In [4]:
from transformers import pipeline

MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
clf = pipeline("text-classification", model=MODEL, tokenizer=MODEL)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
clf(df.loc[0, "tweet_hf"])

[{'label': 'neutral', 'score': 0.7226216197013855}]

#### Inferencia por batches (lotes)

In [8]:
from tqdm import tqdm

def hf_predict_batch(texts, clf, batch_size=128):
    outputs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        outputs.extend(clf(batch))
    return outputs

texts = df["tweet_hf"].tolist()
preds = hf_predict_batch(texts, clf, batch_size=128)


100%|███████████████████████████████████████████████████████████████████████████| 6728/6728 [21:50:49<00:00, 11.69s/it]


In [9]:
df["sentiment_label_hf"] = [p["label"] for p in preds]
df["sentiment_score_hf"] = [p["score"] for p in preds]

In [10]:
df["sentiment_label_hf"].value_counts(normalize=True)

sentiment_label_hf
neutral     0.701680
positive    0.227847
negative    0.070473
Name: proportion, dtype: float64

In [11]:
df.to_parquet("tweets_ai_sentiment_hf.parquet", index=False)